# Detroit Redlining Legacy Analysis

> **Research Question**: Does a neighborhood's 1939 HOLC grade still predict its residents' income and racial composition today, 80 years later?

In the 1930s, the federal Home Owners' Loan Corporation (HOLC) graded American neighborhoods A–D based largely on racial composition. Grade D ("Hazardous") neighborhoods were outlined in red, giving rise to the term **redlining**. These grades determined access to government-backed mortgages, shaping generational wealth for decades.

This project quantifies that legacy using:
- 1939 HOLC district boundaries (University of Richmond DSL)
- 2018 ACS median household income (U.S. Census Bureau API)
- 2018 ACS racial composition (U.S. Census Bureau API)

---
**Skills demonstrated**: JSON parsing · REST API integration · ETL pipeline · geospatial data · statistical analysis · data visualization

## 0. Setup

In [ ]:
import json, os, re, random, time, requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Polygon as MplPolygon
from matplotlib.path import Path
from collections import Counter

# Optional: interactive charts
try:
    import plotly.express as px
    import folium
    INTERACTIVE = True
except ImportError:
    INTERACTIVE = False
    print('Run: pip install plotly folium  to enable interactive charts')

random.seed(17)

GRADE_COLORS = {'A':'darkgreen','B':'cornflowerblue','C':'gold','D':'maroon'}
GRADE_LABELS = {'A':'Best','B':'Still Desirable','C':'Definitely Declining','D':'Hazardous'}
CENSUS_KEY   = "YOUR_KEY_HERE"   # https://api.census.gov/data/key_signup.html
CACHE_FILE   = 'redlines_cache_portfolio.json'
DATA_FILE    = 'redlines_data.json'

os.makedirs('outputs', exist_ok=True)
print('Setup complete.')

---
## 1. Extract — Parse GeoJSON

The raw data is a GeoJSON file with 238 HOLC-graded district polygons.
Each feature contains:
- `geometry.coordinates` — polygon boundary (lon, lat pairs)
- `properties.holc_grade` — A / B / C / D
- `properties.area_description_data['8']` — qualitative appraisal text written in 1939

**Key engineering decision**: Some districts are `MultiPolygon` (non-contiguous land parcels). We select the largest ring by point count to get a representative boundary.

In [ ]:
def parse_districts(file_name):
    """Parse GeoJSON → list of district dicts. Handles Polygon + MultiPolygon."""
    with open(file_name) as f:
        raw = json.load(f)

    districts = []
    for feature in raw['features']:
        geom_type = feature['geometry']['type']
        coords    = feature['geometry']['coordinates']

        # Flatten to a single representative ring
        if geom_type == 'Polygon':
            ring = coords[0]
        else:  # MultiPolygon
            all_rings = [r for poly in coords for r in poly]
            ring = max(all_rings, key=len)   # largest ring = main polygon

        props = feature['properties']
        grade = props['holc_grade']
        districts.append({
            'coordinates': ring,
            'holcGrade'  : grade,
            'holcColor'  : GRADE_COLORS.get(grade, 'gray'),
            'id'         : props['holc_id'],
            'description': props.get('area_description_data', {}).get('8', ''),
            'randomLat'  : None, 'randomLong': None,
            'censusTract': None, 'medIncome' : None,
            'pctBlack'   : None, 'rank'      : None,
        })
    return districts

districts = parse_districts(DATA_FILE)
print(f"Parsed {len(districts)} districts")

# Grade distribution
from collections import Counter
grade_counts = Counter(d['holcGrade'] for d in districts)
print("Grade distribution:", dict(sorted(grade_counts.items())))

---
## 2. Transform — Enrich with Census Data

To link 1939 district boundaries to modern Census data, we need a 3-step enrichment pipeline:

```
District polygon
      ↓  (mesh grid + Path.contains_points)
Random interior point (lat, lon)
      ↓  (FCC Area API)
Census tract FIPS code (11-digit)
      ↓  (ACS 5-Year API)
Median income + racial composition
```

**Why not use the polygon centroid?** Some districts are non-convex or non-contiguous — the geometric centroid can fall outside the polygon. Sampling from an interior mesh guarantees the point is actually inside.

In [ ]:
def generate_rand_points(districts):
    """
    Build a mesh grid over Detroit's bounding box.
    For each district, find grid points inside its polygon boundary
    and pick one at random as the representative location.
    """
    xgrid = np.arange(-83.5, -82.8, 0.004)  # longitude range
    ygrid = np.arange(42.1,  42.6,  0.004)  # latitude range
    xmesh, ymesh = np.meshgrid(xgrid, ygrid)
    points = np.vstack((xmesh.flatten(), ymesh.flatten())).T  # (N, 2) array

    for d in districts:
        path  = Path(d['coordinates'])          # matplotlib polygon path
        mask  = path.contains_points(points)    # boolean array
        valid = points[mask]                    # points inside the polygon
        if valid.size > 0:
            pt = random.choice(valid)
            d['randomLong'] = float(pt[0])
            d['randomLat']  = float(pt[1])
    print('Random interior points generated.')


def fetch_census_tracts(districts):
    """
    FCC Area API: (lat, lon) → 15-digit block FIPS.
    We take the first 11 digits = state(2) + county(3) + tract(6).
    Retry up to 5× per district to handle intermittent FCC downtime.
    """
    base = "https://geo.fcc.gov/api/census/area"
    for i, d in enumerate(districts):
        if d['randomLat'] is None:
            continue
        params = {'lat': d['randomLat'], 'lon': d['randomLong'],
                  'censusYear': 2010, 'format': 'json'}
        for attempt in range(5):
            try:
                r = requests.get(base, params=params, timeout=10)
                if r.status_code == 200:
                    fips = r.json()['results'][0]['block_fips']
                    d['censusTract'] = fips[:11]   # 11-digit FIPS
                    break
            except Exception:
                time.sleep(1)
        if i % 50 == 0:
            print(f'  FCC lookup: {i}/238')
    print('Census tract lookup complete.')


def fetch_income(districts, key):
    """
    ACS 5-Year 2018, variable B19013_001E = median household income.
    Single API call for all Michigan tracts → match by FIPS.
    """
    r = requests.get("https://api.census.gov/data/2018/acs/acs5", params={
        "get": "B19013_001E", "for": "tract:*",
        "in": "state:26 county:*", "key": key
    }, timeout=30)
    rows   = r.json()
    lookup = {row[1]+row[2]+row[3]: row[0] for row in rows[1:]}   # FIPS → income
    for d in districts:
        if d['censusTract']:
            val = lookup.get(d['censusTract'])
            d['medIncome'] = int(val) if val and val not in ('null','-666666666') and int(val) > 0 else 0
    print('Income data matched.')


def fetch_race(districts, key):
    """
    ACS variables:
      B01003_001E = total population
      B02009_001E = Black or African American population
    pctBlack = Black / total * 100
    """
    r = requests.get("https://api.census.gov/data/2018/acs/acs5", params={
        "get": "B01003_001E,B02009_001E", "for": "tract:*",
        "in": "state:26 county:*", "key": key
    }, timeout=30)
    rows   = r.json()
    lookup = {}
    for row in rows[1:]:
        total = int(row[0]) if row[0] else 0
        black = int(row[1]) if row[1] else 0
        lookup[row[2]+row[3]+row[4]] = round(black/total*100, 2) if total > 0 else 0.0
    for d in districts:
        if d['censusTract']:
            d['pctBlack'] = lookup.get(d['censusTract'])
    print('Racial composition matched.')

In [ ]:
# ── Load from cache (fast) OR run full pipeline (first time, ~5 min) ──
if os.path.exists(CACHE_FILE):
    with open(CACHE_FILE) as f:
        districts = json.load(f)
    print(f'Loaded {len(districts)} districts from cache.')
else:
    print('Running full pipeline (~5 min)...')
    generate_rand_points(districts)
    fetch_census_tracts(districts)
    fetch_income(districts, CENSUS_KEY)
    fetch_race(districts, CENSUS_KEY)
    # Rank by income
    ranked = sorted([d for d in districts if d['medIncome'] and d['medIncome'] > 0],
                    key=lambda x: x['medIncome'], reverse=True)
    for i, d in enumerate(ranked, 1):
        d['rank'] = i
    # Cache results
    with open(CACHE_FILE, 'w') as f:
        json.dump(districts, f, indent=2)
    print(f'Cached → {CACHE_FILE}')

# Quick validation
has_income = sum(1 for d in districts if d['medIncome'] and d['medIncome'] > 0)
has_race   = sum(1 for d in districts if d['pctBlack'] is not None)
print(f'Districts with income data: {has_income}/238')
print(f'Districts with race data:   {has_race}/238')

---
## 3. Analyse — Income & Racial Composition by Grade

In [ ]:
# Build summary DataFrame
df = pd.DataFrame(districts)
df = df[df['medIncome'] > 0].copy()

summary = df.groupby('holcGrade').agg(
    n         = ('medIncome', 'count'),
    mean_inc  = ('medIncome', 'mean'),
    median_inc= ('medIncome', 'median'),
    mean_pct_black = ('pctBlack', 'mean')
).reindex(['A','B','C','D']).round(0)

summary.index = [f"Grade {g} — {GRADE_LABELS[g]}" for g in summary.index]
print(summary.to_string())

# Key stats
a_mean = df[df['holcGrade']=='A']['medIncome'].mean()
d_mean = df[df['holcGrade']=='D']['medIncome'].mean()
print(f"\n→ Grade A earns {a_mean/d_mean:.1f}× more than Grade D")
print(f"→ Raw income gap: ${a_mean - d_mean:,.0f} per year")

---
## 4. Text Analysis — What Language Did HOLC Appraisers Use?

The 1939 appraisal descriptions reveal how appraisers perceived neighborhoods. We extract words **unique to each grade** — words that appear in one grade's descriptions but not others.

In [ ]:
FILLER = {'the','of','and','in','to','a','is','for','on','that','are',
           'as','with','this','by','have','from','at','an','it','be',
           'was','been','will','its','or','which','has'}

# Aggregate descriptions by grade
texts  = {g: ' '.join(d['description'].lower() for d in districts
                       if d['holcGrade'] == g and d['description']) for g in 'ABCD'}
tokens = {g: [w for w in re.findall(r'\b[a-z]+\b', t) if w not in FILLER]
           for g, t in texts.items()}
counts = {g: Counter(tok) for g, tok in tokens.items()}

# Keep only words unique to each grade
exclusive = {}
for g in 'ABCD':
    others = set(w for og, c in counts.items() if og != g for w in c)
    exclusive[g] = Counter({w: n for w, n in counts[g].items() if w not in others})

print("Top 5 distinctive words per grade:")
for g in 'ABCD':
    top = ', '.join(w for w, _ in exclusive[g].most_common(5))
    print(f"  Grade {g} ({GRADE_LABELS[g]}): {top}")

---
## 5. Visualise

Five charts telling the story from different angles.

In [ ]:
# ── Chart 1: 1939 HOLC District Map ──
fig, ax = plt.subplots(figsize=(13, 11))
for d in districts:
    poly = MplPolygon(d['coordinates'], closed=True,
                      facecolor=d['holcColor'], edgecolor='black', linewidth=0.3)
    ax.add_patch(poly)
ax.autoscale()
ax.set_title("1939 Detroit HOLC Redlining Map", fontsize=16, fontweight='bold')
ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
legend_patches = [mpatches.Patch(color=c, label=f"Grade {g} — {GRADE_LABELS[g]}")
                  for g, c in GRADE_COLORS.items()]
ax.legend(handles=legend_patches, loc='lower right', fontsize=10)
plt.tight_layout()
plt.savefig('outputs/01_holc_map.png', dpi=150)
plt.show()

In [ ]:
# ── Chart 2: Income Distribution by Grade (Box Plot) ──
fig, ax = plt.subplots(figsize=(10, 6))
data_by_grade = {g: [d['medIncome'] for d in districts
                     if d['holcGrade']==g and d['medIncome'] and d['medIncome']>0]
                 for g in 'ABCD'}
bp = ax.boxplot([data_by_grade[g] for g in 'ABCD'],
                patch_artist=True, widths=0.5)
for patch, g in zip(bp['boxes'], 'ABCD'):
    patch.set_facecolor(GRADE_COLORS[g]); patch.set_alpha(0.8)
ax.set_xticklabels([f"Grade {g}\n{GRADE_LABELS[g]}" for g in 'ABCD'])
ax.set_ylabel("2018 Median Household Income (USD)", fontsize=12)
ax.set_title("Income Distribution by 1939 HOLC Grade\n"
             "Redlined neighborhoods remain significantly poorer 80 years later",
             fontsize=13, fontweight='bold')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f"${x:,.0f}"))
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('outputs/02_income_boxplot.png', dpi=150)
plt.show()

In [ ]:
# ── Chart 3: Race vs Income Scatter (KEY CHART) ──
fig, ax = plt.subplots(figsize=(11, 7))
plot_data = [d for d in districts if d['medIncome'] and d['medIncome']>0
             and d['pctBlack'] is not None]
for d in plot_data:
    ax.scatter(d['pctBlack'], d['medIncome'],
               c=d['holcColor'], s=60, alpha=0.75, edgecolors='white', linewidths=0.5)
xs = [d['pctBlack'] for d in plot_data]
ys = [d['medIncome'] for d in plot_data]
m, b = np.polyfit(xs, ys, 1)
xfit = np.linspace(min(xs), max(xs), 100)
ax.plot(xfit, m*xfit+b, 'k--', linewidth=1.5, label=f'Trend (slope: ${m:,.0f}/pct pt)')
legend_patches = [mpatches.Patch(color=c, label=f"Grade {g} — {GRADE_LABELS[g]}")
                  for g, c in GRADE_COLORS.items()]
ax.legend(handles=legend_patches, fontsize=9)
ax.set_xlabel("% Black / African-American Residents (2018 ACS)", fontsize=12)
ax.set_ylabel("Median Household Income, 2018 (USD)", fontsize=12)
ax.set_title("Racial Composition vs. Household Income by HOLC Grade\n"
             f"Every 1% increase in Black residents → ${abs(m):,.0f} lower median income",
             fontsize=12, fontweight='bold')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f"${x:,.0f}"))
ax.grid(alpha=0.2)
plt.tight_layout()
plt.savefig('outputs/03_race_income_scatter.png', dpi=150)
plt.show()
print(f"Regression slope: ${m:,.0f} per 1% increase in Black residents")

In [ ]:
# ── Chart 4: Distinctive Words by Grade ──
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, g in zip(axes.flatten(), 'ABCD'):
    top = exclusive[g].most_common(10)
    words, freqs = zip(*top) if top else ([], [])
    ax.barh(list(words)[::-1], list(freqs)[::-1],
            color=GRADE_COLORS[g], alpha=0.85)
    ax.set_title(f"Grade {g} — {GRADE_LABELS[g]}",
                 fontweight='bold', color=GRADE_COLORS[g])
    ax.set_xlabel("Frequency")
    ax.grid(axis='x', alpha=0.3)
fig.suptitle("Distinctive Words in 1939 HOLC Appraisal Descriptions\n"
             "Grade D language: 'unreliable', 'slum', 'tenants' — racial bias encoded in text",
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/04_distinctive_words.png', dpi=150)
plt.show()

In [ ]:
# ── Chart 5: Interactive Map (if folium available) ──
if INTERACTIVE:
    m = folium.Map(location=[42.35, -83.1], zoom_start=11, tiles='CartoDB positron')
    for d in districts:
        latlon = [[pt[1], pt[0]] for pt in d['coordinates']]
        income_str = f"${d['medIncome']:,}" if d['medIncome'] else 'N/A'
        pct_str    = f"{d['pctBlack']:.1f}%" if d['pctBlack'] is not None else 'N/A'
        popup_html = (f"<b>District {d['id']}</b> — Grade {d['holcGrade']}<br>"
                      f"2018 Median Income: {income_str}<br>"
                      f"% Black residents: {pct_str}")
        folium.Polygon(
            locations=latlon, color='black', weight=0.5,
            fill=True, fill_color=d['holcColor'], fill_opacity=0.65,
            popup=folium.Popup(popup_html, max_width=250)
        ).add_to(m)
    m.save('outputs/05_interactive_map.html')
    print('Interactive map saved → outputs/05_interactive_map.html')
    display(m)
else:
    print('Install folium for interactive map: pip install folium')

---
## 6. Key Findings Summary

In [2]:
import pandas as pd

summary_findings = {
    'Finding': [
        'Grade A vs D income ratio',
        'Raw annual income gap',
        'Income drop per 1% Black residents',
        'Grade D avg % Black residents',
        'Grade A avg % Black residents',
    ],
    'Value': [
        '3.3x',
        '$68,213 / year',
        '$503',
        '60.7%',
        '38.6%',
    ]
}

pd.DataFrame(summary_findings).set_index('Finding').style\
    .set_properties(**{'text-align': 'left', 'font-size': '13px'})\
    .set_table_styles([
        {'selector': 'th', 'props': [('font-size', '13px'), ('text-align', 'left')]},
        {'selector': 'tr:nth-child(even)', 'props': [('background-color', '#f5f5f5')]}
    ])

,Value
Finding,
Grade A vs D income ratio,3.3x
Raw annual income gap,"$68,213 / year"
Income drop per 1% Black residents,$503
Grade D avg % Black residents,60.7%
Grade A avg % Black residents,38.6%


### Business Implications

This analysis is directly relevant to:
- **Fintech / Lending**: Detecting modern algorithmic redlining in mortgage approval models
- **Real estate analytics**: Quantifying neighbourhood investment gaps from historical policy  
- **ESG / Policy reporting**: Measuring legacy discrimination for fair housing compliance
- **Urban planning**: Identifying areas where targeted investment could close income gaps

### Limitations & Next Steps
- Census tract boundaries have changed since 1939, some boundary mismatch is inevitable
- Analysis is correlational, not causal, other variables (proximity to industry, school quality) also matter
- Next step: run a multivariate regression controlling for confounders to isolate the HOLC grade effect